In [52]:
from dotenv import load_dotenv

load_dotenv()

True

In [53]:
import re
from typing import Any
from dataclasses import dataclass
from pathlib import Path
import pandas as pd

from langchain.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.agents import create_agent
from langchain.messages import HumanMessage

In [54]:
#Különböző toolok: calculator, csv_analyzer, search_document

In [55]:
@dataclass(frozen=True)
class AppConfig:
    csv_path: Path
    pdf_path: Path
    embedding_model: str = "text-embedding-3-small"
    retrieval_k: int = 4
    chunk_size: int = 1000
    chunk_overlap: int = 200
    allowed_operations: tuple[str, ...] = (
    "row_count",
    "column_list",
    "head",
    "mean",
    "sum",
    "min",
    "max")

@dataclass
class AppContext:
    config: AppConfig
    df: pd.DataFrame
    vector_store: Any

In [56]:

def build_vector_store(config: AppConfig):
    loader = PyPDFLoader(str(config.pdf_path))
    pages = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.chunk_size,
        chunk_overlap=config.chunk_overlap,
        add_start_index=True,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks = text_splitter.split_documents(pages)

    embeddings = OpenAIEmbeddings(model=config.embedding_model)
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(documents=chunks)
    return vector_store

def build_context(config: AppConfig) -> AppContext:
    df = pd.read_csv(config.csv_path)
    vector_store = build_vector_store(config)

    return AppContext(
        config=config,
        df=df,
        vector_store=vector_store,
    )

In [63]:
def make_calculate_tool():
    @tool
    def calculate(expression: str) -> dict:
        """Matematikai számításokhoz"""

        try:
            expression = expression.strip()

            if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
                return {
                    "success": False,
                    "error": "Invalid characters in expression.",
                    "expression": expression
                }

            result = eval(expression, {"__builtins__": {}}, {})

            return {
                "success": True,
                "result": result,
                "expression": expression
            }

        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "expression": expression
            }
    return calculate

def make_analyze_csv_tool(context: AppContext):
    numeric_operations = {"mean", "sum", "min", "max"}
    @tool
    def analyze_csv(operation: str, column: str = None, rows: int = 5) -> dict:
        """A betöltött CSV elemzéséhez"""
       
        try:
            df = context.df
            allowed_operations = context.config.allowed_operations

            if operation not in allowed_operations:
                raise ValueError(
                    f"Ismeretlen művelet. Elérhető műveletek: {allowed_operations}"
                )

            # numerikus műveletek
            if operation in numeric_operations:

                if column is None:
                    raise ValueError("A numerikus művelethez kell oszlopnév.")

                if column not in df.columns:
                    raise ValueError(
                        f"Az oszlop nem található. Elérhető oszlopok: {df.columns.tolist()}"
                    )

                if not pd.api.types.is_numeric_dtype(df[column]):
                    raise ValueError("Az oszlop nem numerikus.")

                result = getattr(df[column], operation)()

            elif operation == "row_count":
                result = len(df)

            elif operation == "column_list":
                result = df.columns.tolist()

            elif operation == "head":
                result = df.head(rows).to_dict(orient="records")

            # numpy típus konvertálás python natívra
            if hasattr(result, "item"):
                result = result.item()

            return {
                "success": True,
                "operation": operation,
                "result": result
            }

        except Exception as e:
            return {
                "success": False,
                "operation": operation,
                "error": str(e)
            }
    return analyze_csv

def make_search_document_tool(context: AppContext):
    @tool(response_format="content_and_artifact")
    def search_document(query: str):
        """A PDF dokumentumban kereséshez"""
        retrieved_docs = context.vector_store.similarity_search(query, k=context.config.retrieval_k)
        if not retrieved_docs:
            return "Nem találtam releváns részt a dokumentumban.", []
        serialized = "\n\n".join(
            (f"Oldal: {doc.metadata.get('page_label', doc.metadata.get('page', 'N/A'))}\nTartalom: {doc.page_content}")
            for doc in retrieved_docs
        )
        return serialized, retrieved_docs
    return search_document

In [64]:
def make_tools(context: AppContext):
    calculate = make_calculate_tool()
    analyze_csv = make_analyze_csv_tool(context)
    search_document = make_search_document_tool(context)
    return [calculate, analyze_csv, search_document]

In [65]:
SYSTEM_PROMPT = """
Te egy dokumentumelemző és adózási agent vagy
- Dokumentumkérdéshez használd a search_document toolt
- CSV kérdéshez használd az analyze_csv toolt
- Számításhoz használd a calculate toolt
- Magyarul, tömören válaszolj
- Ne sorold fel az eszközeidet
- Ne kérdezz vissza
"""

def build_agent(config: AppConfig):
    context = build_context(config)
    tools = make_tools(context)

    agent = create_agent(
        model="gpt-5-nano",
        tools=tools,
        system_prompt=SYSTEM_PROMPT
    )
    return agent, context, tools

In [66]:
config = AppConfig(
    csv_path=Path("../data/sales.csv"),
    pdf_path=Path("../data/kisvallalati_ado_szabalyzat.pdf"),
)


multi_tool_agent, context, tools = build_agent(config)

calculate, analyze_csv, search_document = tools

In [67]:
# Tool smoke testek

print(calculate.invoke({"expression": "2**10"}))
print(analyze_csv.invoke({"operation": "row_count"}))
print(analyze_csv.invoke({"operation": "column_list"}))
print(search_document.invoke({"query": "Hogyan lehet kikerülni a kisvállalati adó hatálya alól?"}))

{'success': True, 'result': 1024, 'expression': '2**10'}
{'success': True, 'operation': 'row_count', 'result': 4}
{'success': True, 'operation': 'column_list', 'result': ['month', 'revenue', 'profit', 'region']}
Oldal: 14
Tartalom: 14 
 
Felhívjuk a figyelmet, hogy az átalakulás, egyesülés, szétválás útján jogutódként létrejövő adózó a 
kisvállalati adó szempontjából nem számít tevékenységét kezdő vállalkozásnak, így a kisvállalatiadó -
alanyiság választását a már működő adózókra vonatkozó szabályok alapján teheti meg.  
5. Kikerülés a kisvállalati adó hatálya alól 
A kisvállalati adó szerinti adóalanyiság főszabály szerint azon adóév utolsó napjáig áll fenn, amely 
adóévben az adóalany az erre a célra rendszeresített nyomtatványon, elektronikusan bejelenti NAV-hoz, 
hogy adókötelezettségeit nem a kisvállalati adó szabályai szerint teljesíti. A bejelentést legkorábban az 
adóév december 1-jétől, legkésőbb az adóév december 20 -áig lehet megtenni. A határidő elmulasztása 
esetén igazolá

In [68]:
from pprint import pprint

response = multi_tool_agent.invoke(
    {"messages":[HumanMessage(content="Mennyi 50000 forint 27% áfával?")]}
)

pprint(response)

{'messages': [HumanMessage(content='Mennyi 50000 forint 27% áfával?', additional_kwargs={}, response_metadata={}, id='36cdf678-c74b-4ad2-9e86-89c0265fb536'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 540, 'prompt_tokens': 295, 'total_tokens': 835, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWkVU7lILJ67lbJuJbsnS3AwYvK2L', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dab6e-844e-7bc0-a7c7-008cf1946609-0', tool_calls=[{'name': 'calculate', 'args': {'expression': '50000 * 1.27'}, 'id': 'call_VlU5L8nzpIoWZ7fvXJUEmFju', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to

In [69]:
response = multi_tool_agent.invoke(
    {"messages":[HumanMessage(content="Mi a revenue maximuma a CSV-ben?")]}
)

pprint(response)

{'messages': [HumanMessage(content='Mi a revenue maximuma a CSV-ben?', additional_kwargs={}, response_metadata={}, id='395579c9-accc-4e44-a551-2be7de24c8c0'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 412, 'prompt_tokens': 289, 'total_tokens': 701, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWkVbSwziPZsM17QCBLXGgndMK3sY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dab6e-a019-7833-9442-84e5da8fe935-0', tool_calls=[{'name': 'analyze_csv', 'args': {'operation': 'max', 'column': 'Revenue'}, 'id': 'call_gxBDz8HgZGxtlz4WzMxNATXE', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metad

In [70]:
response = multi_tool_agent.invoke(
    {"messages":[HumanMessage(content="Mit ír a dokumentum az ÁFA levonhatóságáról?")]}
)

pprint(response)

{'messages': [HumanMessage(content='Mit ír a dokumentum az ÁFA levonhatóságáról?', additional_kwargs={}, response_metadata={}, id='32bc6743-107e-49b6-b768-ab7720711c74'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 222, 'prompt_tokens': 296, 'total_tokens': 518, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWkVmbNUNc0WxbzVOHbBS2yaSzzjP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dab6e-ce39-7fd1-b422-1292cf671e5e-0', tool_calls=[{'name': 'search_document', 'args': {'query': 'ÁFA levonhatóság'}, 'id': 'call_6Ey2zuIeEXlQtjMaCDkqvZRp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m